### P2.3 — Deep Learning Foundations | Python to Gen AI Course
### P2.3.3 — From Neuron to Network

---
## 🔁 Quick Recap — Where We Are

In P2.3.2 we built a single neuron from scratch:

```
Inputs  →  Weights  →  Weighted Sum + Bias  →  Activation  →  Output
```

We saw all four parts — inputs, weights, bias, activation function.  
We coded it in pure Python. We ran it. It worked.

But we ended with one honest observation:

> One neuron — even with an activation function — is still just **one step of thinking.**  
> It can only do one check on the data.  
> Real problems need many checks, combined together.

So the natural next question is: **what happens when we connect many neurons?**

That is exactly what this notebook answers.

---
## 🏗️ Stacking Neurons Into Layers

Instead of one neuron, imagine a **column of neurons** all receiving the same inputs.

Each neuron in the column:
- Has its own set of weights
- Learns a different pattern from the same input
- Produces its own output

```
                    Neuron A  →  detected: edges
Input  ──────────▶  Neuron B  →  detected: corners
                    Neuron C  →  detected: curves
```

This column of neurons is called a **Layer.**

And when you stack multiple layers — each one building on the previous —  you get a **Neural Network.**

<img src="images/neurons_to_layer.png" width="750"/>


---
## 🗂️ The 3 Types of Layers

Every neural network has exactly 3 types of layers:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   INPUT LAYER   │ ──▶ │  HIDDEN LAYERS  │ ──▶ │  OUTPUT LAYER   │
│                 │     │  (1 or many)    │     │                 │
│ Receives the    │     │ Learns patterns │     │ Produces the    │
│ raw data        │     │ from the data   │     │ final answer    │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

### Input Layer
- One input per feature in your data
- Does no computation — just passes the raw data into the network
- House price example: 3 inputs (size, beds, age)

### Hidden Layers
- This is where all the learning happens
- Each neuron picks up a different aspect of the data
- Every neuron's output travels to **every neuron in the next layer** — each with its own weight
- You choose how many hidden layers and how many neurons per layer

### Output Layer
- Produces the final answer
- The number of output neurons depends on what you're predicting:

```
Predicting a number         →  1 neuron   (e.g. house price)
Yes or No answer            →  1 neuron   (e.g. spam or not spam)
One of many options         →  N neurons  (e.g. cat / dog / bird)
```

<img src="images/three_layer_types.png" width="800"/>

---

<img src="images/three_layer_types2.png" width="800"/>

---
## ➡️ The Forward Pass — How Data Flows Through

When you feed data into a network, it travels **forward** through every layer —  
from input to output. This is called the **Forward Pass.**

At each layer:
```
Step 1 — Take outputs from the previous layer as inputs
Step 2 — Multiply each input by its weight
Step 3 — Sum them up and add bias
Step 4 — Pass through activation function
Step 5 — Send output to the next layer
```

This repeats at every layer until the output layer produces a prediction.

<img src="images/forward_pass.png" width="600"/>

---
## 💻 Let's Build It — Forward Pass in Pure Python

Before we use any framework, let's manually build a small network  
and trace exactly what happens at each layer.

Network: 3 inputs → 1 hidden layer (4 neurons) → 1 output neuron

In [1]:
import math

# ── NETWORK SHAPE ─────────────────────────────────────────────────
# Input Layer  : 3 inputs  (size, beds, age)
# Hidden Layer : 1 layer   with 4 neurons
# Output Layer : 1 neuron  (single prediction)
# Total connections: (3×4) + (4×1) = 12 + 4 = 16 weights

# ── ACTIVATION FUNCTIONS ──────────────────────────────────────────
def relu(z):
    return max(0, z)

def sigmoid(z):
    return 1 / (1 + math.exp(-z))


# ── INPUT LAYER ───────────────────────────────────────────────────
# House: 1600 sqft, 3 beds, 4 years old
inputs = [1600, 3, 4]
print(f"Input Layer  : {inputs}")


# ── HIDDEN LAYER — 4 neurons ──────────────────────────────────────
# Each neuron has its own weights for each input
# In a real network these are learned — here we set them manually
hidden_weights = [
    [0.5,  0.3, -0.2],   # Neuron 1 weights
    [0.8, -0.1,  0.4],   # Neuron 2 weights
    [-0.3, 0.7,  0.6],   # Neuron 3 weights
    [0.2,  0.5, -0.5],   # Neuron 4 weights
]
hidden_biases = [0.1, 0.2, -0.1, 0.3]

# Forward pass through hidden layer
hidden_outputs = []
for i, (weights, bias) in enumerate(zip(hidden_weights, hidden_biases)):
    # Weighted sum
    z = sum(x * w for x, w in zip(inputs, weights)) + bias
    # ReLU activation
    activated = relu(z)
    hidden_outputs.append(activated)
    print(f"  Hidden Neuron {i+1}: z={round(z,2):>8}  →  ReLU  →  {round(activated,2)}")

print(f"Hidden Layer Output : {[round(h, 2) for h in hidden_outputs]}")


# ── OUTPUT LAYER — 1 neuron ───────────────────────────────────────
# Takes the hidden layer outputs as its inputs
output_weights = [0.6, -0.4, 0.9, 0.3]
output_bias    = 0.5

z_out  = sum(h * w for h, w in zip(hidden_outputs, output_weights)) + output_bias
output = sigmoid(z_out)

print(f"\nOutput Layer: z={round(z_out, 2)}  →  Sigmoid  →  {round(output, 4)}")
print(f"\nFinal Prediction: {round(output, 4)}")

Input Layer  : [1600, 3, 4]
  Hidden Neuron 1: z=   800.2  →  ReLU  →  800.2
  Hidden Neuron 2: z=  1281.5  →  ReLU  →  1281.5
  Hidden Neuron 3: z=  -475.6  →  ReLU  →  0
  Hidden Neuron 4: z=   319.8  →  ReLU  →  319.8
Hidden Layer Output : [800.2, 1281.5, 0, 319.8]

Output Layer: z=63.96  →  Sigmoid  →  1.0

Final Prediction: 1.0


---
## 🔍 What Just Happened?

```
inputs = [1600, 3, 4]
         │
         ▼
Hidden Layer (4 neurons, each with own weights)
  Neuron 1 → weighted sum → ReLU → output
  Neuron 2 → weighted sum → ReLU → output
  Neuron 3 → weighted sum → ReLU → output
  Neuron 4 → weighted sum → ReLU → output
         │
         ▼
Output Layer (1 neuron, takes hidden outputs as input)
  → weighted sum → Sigmoid → final prediction
```

We manually traced every number through every layer.  

In a real network this happens millions of times per second —  across thousands of neurons — but the math is identical.

---
## ❓ How Do You Decide the Network Shape?

When building a network, you make 3 decisions:

```
Decision 1 — How many hidden layers?
  Simple problem   →  1–2 layers
  Complex problem  →  3–10+ layers
  Images / Speech  →  can go very deep (50–100+ layers)

Decision 2 — How many neurons per hidden layer?
  Common starting points: 64, 128, 256
  More neurons = more capacity to learn
  Too many and the network starts memorising instead of learning

Decision 3 — How many output neurons?
  Predicting a number         →  1
  Yes or No answer            →  1
  One of N options            →  N
```

> There is no single right answer.  
> You start with a reasonable shape, train it, check the results,  
> and adjust. This is normal in Deep Learning.

<img src="images/network_shapes.png" width="750"/>

---
## ✅ Summary — What You Learned

| Concept | Key Point |
|---|---|
| Layer | A column of neurons all receiving the same inputs |
| Input Layer | One input per feature — no computation, just passes raw data in |
| Hidden Layers | Where learning happens — each neuron picks up a different aspect of the data |
| Fully Connected | Every neuron's output travels to every neuron in the next layer — each with its own weight |
| Output Layer | Produces the final answer — 1 neuron for a number or yes/no, N neurons for N options |
| Forward Pass | Data flows layer by layer from input to output — same 5 steps at every layer |
| Network Shape | You decide: how many hidden layers, how many neurons per layer, how many outputs |